# Data Loading (Local-First)

This notebook is a clean, fast local workflow for Task 1.
It avoids heavy model training and focuses on:
- chunked loading
- memory optimization
- parquet conversion
- EDA-ready outputs

In [ ]:
# Imports and project path setup
from pathlib import Path
import sys
import pandas as pd

project_root = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(project_root))

from src.utils.config import get_config, get_system_info
from src.data.memory_utils import compare_memory, optimize_dtypes
from src.data.preprocessing import (
    build_city_traffic_dataframe,
    save_city_parquet,
    load_city_parquet,
    get_top_square_id,
    get_area_series,
    )

cfg = get_config()
get_system_info()

SyntaxError: '(' was never closed (3110041220.py, line 11)

In [ ]:
# Paths
raw_dir = cfg.data_raw
parquet_path = cfg.data_processed / "city_traffic.parquet"
raw_files = sorted(raw_dir.glob("sms-call-internet-mi-*.txt"))
len(raw_files), raw_files[:2]

In [ ]:
# Memory optimization probe on a small sample
sample_file = raw_files[0]
raw = pd.read_csv(
    sample_file,
    sep="\t",
    header=None,
    names=[
        "square_id",
        "time_interval",
        "country_code",
        "sms_in",
        "sms_out",
        "call_in",
        "call_out",
        "internet_traffic",
    ],
    usecols=[0, 1, 7],
    nrows=300_000,
 )
optimized = optimize_dtypes(raw)
cmp = compare_memory(raw, optimized)
{
    "before_mb": round(cmp.before_mb, 2),
    "after_mb": round(cmp.after_mb, 2),
    "reduction_pct": round(cmp.reduction_pct, 2),
}

In [ ]:
# Build parquet once (chunked). Skip if it already exists.
if not parquet_path.exists():
    city_df = build_city_traffic_dataframe(raw_dir, chunksize=2_000_000)
    save_city_parquet(city_df, parquet_path)
    print(f"Parquet saved to {parquet_path}")
else:
    print(f"Parquet already exists: {parquet_path}")

In [ ]:
# Load parquet for fast EDA
city_df = load_city_parquet(parquet_path)
city_df.head()

In [ ]:
# Identify the highest-traffic area and check required squares
top_square = get_top_square_id(city_df)
top_square

In [ ]:
# Extract the three target series for Task 2
series_top = get_area_series(city_df, top_square)
series_4159 = get_area_series(city_df, 4159)
series_4556 = get_area_series(city_df, 4556)
series_top.head()

## Next (fast local)
Move to EDA notebook for plots, stationarity, decomposition, and spatial analysis.

## Later (GPU)
Move LSTM/GRU training and long tuning to Kaggle or Colab.